# Water-gas shift degree of rate control

Reproduces Case II of Yang, Achar & Kitchin, *Evaluation of the degree of rate control via automatic differentiation*, AIChE J. **68**(6):e17653 (2022).

The WGS net rate (~1e-6 s⁻¹) is a ~12-order cancellation of one-way step rates (~1e5-1e6 s⁻¹), so finite differences over a re-solved steady state **cannot** compute this DRC — it must be propagated analytically through the steady-state Jacobian, which is exactly what discopt's implicit-differentiation sensitivity does.

The 7 steps use **explicit** rate constants (`kf`, `Keq`) transcribed from the paper SI, and the stiff steady state is solved in **log coverages** from a numeric warm start.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root for discopt_mkm

## The 7-step mechanism (explicit kf / Keq)

In [2]:
import discopt_mkm as mk
from discopt_mkm import numeric
from discopt_mkm.analysis import degree_of_rate_control
from discopt_mkm.examples import water_gas_shift

m, reactor = water_gas_shift(T=480.0)
for rxn in m.reactions:
    print(f'  {rxn.name:26s}  kf={rxn.kf:.2e}  Keq={rxn.Keq:.2e}')

  S1: CO + * -> CO*           kf=1.33e+08  Keq=2.15e+02
  S2: H2O + * -> H2O*         kf=2.01e+11  Keq=5.93e-05
  S3: H2O* + * -> OH* + H*    kf=2.64e+06  Keq=6.28e-02
  S4: OH* + * -> O* + H*      kf=5.24e+01  Keq=1.18e-05
  S5: CO* + O* -> CO2* + *    kf=2.05e+05  Keq=1.03e+03
  S6: CO2* -> CO2 + *         kf=1.48e+12  Keq=1.92e+05
  S7: 2 H* -> H2 + 2*         kf=5.32e+02  Keq=4.50e+01


## Numeric warm start

Coverages span ~12 orders of magnitude — a pure-NumPy `fsolve` from a physical seed locates the (non-poisoned) physical root.

In [3]:
H2 = m._by_name['H2']
seed = {a: 1e-3 for a in m.adsorbates}
theta0, free0 = numeric.steady_state_numeric(m, reactor.pressures, 480.0, theta0=seed)
for a in m.adsorbates:
    print(f'  theta[{a.name:5s}] = {theta0[a]:.4e}')
print(f'  r_H2 (numeric) = {numeric.turnover_frequency(m, H2, theta0, free0, reactor.pressures, 480.0):.4e}')

  theta[CO*  ] = 9.3315e-01
  theta[H2O* ] = 7.7212e-07
  theta[OH*  ] = 5.2418e-07
  theta[H*   ] = 5.6606e-03
  theta[O*   ] = 9.3104e-12
  theta[CO2* ] = 2.7449e-08
  r_H2 (numeric) = 1.4467e-06


## Log-coverage steady state + DRC

discopt refines the warm start and returns the sensitivity. The DRC sums to 1, with two steps (S4 and S5 in the paper's numbering) carrying ~0.88 and ~0.12.

In [4]:
sol = mk.solve_steady_state(m, reactor, coordinates='log', theta0=theta0, log_box=8.0)
print(f'status={sol.status}, r_H2={sol.production_rate(H2):.6e}')
X = degree_of_rate_control(sol, species=H2)
for rxn, x in X.items():
    mark = '  <--' if abs(x) > 0.05 else ''
    print(f'  {rxn.name:26s}: {x:+.4f}{mark}')
print(f'  sum = {sum(X.values()):.4f}')

status=optimal, r_H2=1.446732e-06
  S1: CO + * -> CO*         : -0.0000
  S2: H2O + * -> H2O*       : +0.0000
  S3: H2O* + * -> OH* + H*  : +0.0000
  S4: OH* + * -> O* + H*    : +0.8838  <--
  S5: CO* + O* -> CO2* + *  : +0.1161  <--
  S6: CO2* -> CO2 + *       : +0.0000
  S7: 2 H* -> H2 + 2*       : +0.0001
  sum = 1.0000
